# IT9203 — Data Processing with Apache Spark

**Project:** Top-Rated Movies per Genre Tracker  
**Platform:** Azure Databricks  
**Notebook:** 04 — Data Processing (10 marks)  
**Input:** DBFS `/FileStore/movielens/raw/merged_ratings/`  
**Output:** DBFS `/FileStore/movielens/processed/` + Hive tables

## 1. Overview

| Step | Action | Output |
|---|---|---|
| 1 | Load merged Parquet from DBFS | `raw_df` |
| 2 | Data quality check | Null/duplicate counts |
| 3 | Data cleaning + genre explode | `cleaned_df` |
| 4 | Movie aggregation per genre | `movie_scores` |
| 5 | Top-10 per genre (window function) | `top_per_genre` |
| 6 | Write to DBFS | Partitioned Parquet |
| 7 | Write to Hive tables | `movielens.ratings_cleaned`, `movielens.top_movies` |

## 2. Load Merged Data from DBFS

In [0]:
from pyspark.sql.functions import (
    col, count, avg, sum, max, min, round, rank, explode, split, when, log1p
)
from pyspark.sql.window import Window
import time

spark.sql("USE movielens")

start_time = time.time()
raw_df = spark.read.parquet("/FileStore/movielens/raw/merged_ratings/")
raw_count = raw_df.count()

print(f"Total records loaded: {raw_count:,}")
print(f"Load time: {time.time() - start_time:.2f}s")
raw_df.printSchema()
display(raw_df.limit(5))

Total records loaded: 25,062,518
Load time: 1.02s
root
 |-- movieId: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- review_date: date (nullable = true)



movieId,userId,rating,timestamp,title,genres,review_date
1961,62423,4.0,1166106008,Rain Man (1988),Drama,2006-12-14
2000,62423,3.0,1164035431,Lethal Weapon (1987),Action|Comedy|Crime|Drama,2006-11-20
2001,62423,3.5,1164035646,Lethal Weapon 2 (1989),Action|Comedy|Crime|Drama,2006-11-20
2002,62423,3.5,1164038255,Lethal Weapon 3 (1992),Action|Comedy|Crime|Drama,2006-11-20
2006,62423,3.0,1164035847,"Mask of Zorro, The (1998)",Action|Comedy|Romance,2006-11-20


## 3. Data Quality Check

In [0]:
print("=== NULL VALUE ANALYSIS ===")
null_counts = raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
])
display(null_counts)

dup_count = raw_count - raw_df.dropDuplicates(["userId", "movieId"]).count()
invalid_ratings = raw_df.filter((col("rating") < 0.5) | (col("rating") > 5.0)).count()

print(f"Duplicate (userId, movieId): {dup_count:,}")
print(f"Invalid ratings:              {invalid_ratings:,}")

=== NULL VALUE ANALYSIS ===


movieId,userId,rating,timestamp,title,genres,review_date
62423,0,62423,62423,62423,62423,62423


Duplicate (userId, movieId): 0
Invalid ratings:              0


## 4. Clean Data and Explode Genres

Movies belong to multiple genres (pipe-delimited). We explode them so each genre gets its own row for per-genre ranking.

In [0]:
cleaned_df = raw_df \
    .dropDuplicates(["userId", "movieId"]) \
    .filter(col("rating").between(0.5, 5.0)) \
    .filter(col("movieId").isNotNull()) \
    .filter(col("title").isNotNull()) \
    .withColumn("genre", explode(split(col("genres"), "\\|")))

cleaned_count = cleaned_df.count()
print(f"Records after cleaning + explode: {cleaned_count:,}")
display(cleaned_df.limit(5))

Records after cleaning + explode: 67,809,886


movieId,userId,rating,timestamp,title,genres,review_date,genre
6709,62423,4.0,1164380821,Once Upon a Time in Mexico (2003),Action|Adventure|Crime|Thriller,2006-11-24,Action
6709,62423,4.0,1164380821,Once Upon a Time in Mexico (2003),Action|Adventure|Crime|Thriller,2006-11-24,Adventure
6709,62423,4.0,1164380821,Once Upon a Time in Mexico (2003),Action|Adventure|Crime|Thriller,2006-11-24,Crime
6709,62423,4.0,1164380821,Once Upon a Time in Mexico (2003),Action|Adventure|Crime|Thriller,2006-11-24,Thriller
3177,62426,1.0,1052503045,Next Friday (2000),Comedy,2003-05-09,Comedy


## 5. Aggregate — Movie Scores per Genre

Formula: `popularity_score = avg(rating) * log(1 + rating_count)`

In [0]:
movie_scores = cleaned_df.groupBy("genre", "movieId", "title") \
    .agg(
        count("*").alias("rating_count"),
        avg("rating").alias("avg_rating"),
        max("review_date").alias("last_rating_date")
    ) \
    .withColumn("popularity_score",
        round(col("avg_rating") * log1p(col("rating_count")), 4)) \
    .filter(col("rating_count") >= 5)

scored_count = movie_scores.count()
print(f"Unique movie-genre combos scored: {scored_count:,}")
print("\nTop 10 by popularity score:")
display(movie_scores.orderBy(col("popularity_score").desc()).limit(10))

Unique movie-genre combos scored: 64,473

Top 10 by popularity score:


genre,movieId,title,rating_count,avg_rating,last_rating_date,popularity_score
Crime,318,"Shawshank Redemption, The (1994)",81482,4.413576004516335,2019-11-21,49.9094
Drama,318,"Shawshank Redemption, The (1994)",81482,4.413576004516335,2019-11-21,49.9094
Thriller,296,Pulp Fiction (1994),79672,4.188912039361382,2019-11-21,47.2747
Comedy,296,Pulp Fiction (1994),79672,4.188912039361382,2019-11-21,47.2747
Drama,296,Pulp Fiction (1994),79672,4.188912039361382,2019-11-21,47.2747
Crime,296,Pulp Fiction (1994),79672,4.188912039361382,2019-11-21,47.2747
Drama,858,"Godfather, The (1972)",52498,4.324336165187245,2019-11-21,46.9993
Crime,858,"Godfather, The (1972)",52498,4.324336165187245,2019-11-21,46.9993
Crime,50,"Usual Suspects, The (1995)",55366,4.284353213163313,2019-11-21,46.7926
Thriller,50,"Usual Suspects, The (1995)",55366,4.284353213163313,2019-11-21,46.7926


## 6. Top-10 Movies per Genre (Window Function)

In [0]:
window = Window.partitionBy("genre").orderBy(col("popularity_score").desc())

top_per_genre = movie_scores \
    .withColumn("rank", rank().over(window)) \
    .filter(col("rank") <= 10) \
    .select("genre", "rank", "movieId", "title",
            "rating_count", "avg_rating", "popularity_score")

n_genres = top_per_genre.select("genre").distinct().count()
print(f"Top-10 movies across {n_genres} genres")
display(top_per_genre.orderBy("genre", "rank").limit(20))

Top-10 movies across 20 genres


genre,rank,movieId,title,rating_count,avg_rating,popularity_score
(no genres listed),1,166024,Whiplash (2013),1030,4.210194174757282,29.2115
(no genres listed),2,183869,Hereditary (2018),1313,3.765041888804265,27.0361
(no genres listed),3,176601,Black Mirror,456,4.256578947368421,26.0702
(no genres listed),4,141866,Green Room (2015),1060,3.6537735849056605,25.4557
(no genres listed),5,171495,Cosmos,277,4.3267148014440435,24.3491
(no genres listed),6,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),1687,3.2350326022525193,24.0405
(no genres listed),7,184399,Eighth Grade (2018),544,3.7371323529411766,23.5469
(no genres listed),8,156605,Paterson,571,3.6637478108581436,23.2616
(no genres listed),9,135426,Fantastic Beasts and Where to Find Them 2 (2018),891,3.2643097643097643,22.176
(no genres listed),10,182723,Cosmos: A Spacetime Odissey,191,4.18848167539267,22.0209


## 7. Write Processed Data to DBFS

In [0]:
cleaned_df.write \
    .mode("overwrite") \
    .partitionBy("genre") \
    .parquet("/FileStore/movielens/processed/cleaned/")
print("Cleaned data written")

top_per_genre.write \
    .mode("overwrite") \
    .parquet("/FileStore/movielens/processed/top_per_genre/")
print("Top-per-genre data written")

Cleaned data written
Top-per-genre data written


## 8. Write to Delta Tables

In [0]:
# Write cleaned ratings to Hive
cleaned_df.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("genre") \
    .saveAsTable("movielens.ratings_cleaned")
print("ratings_cleaned table written")

# Write top movies per genre to Hive
top_per_genre.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("movielens.top_movies")
print("top_movies table written")

spark.sql("SHOW TABLES IN movielens").show()

ratings_cleaned table written
top_movies table written
+---------+---------------+-----------+
| database|      tableName|isTemporary|
+---------+---------------+-----------+
|movielens|ratings_cleaned|      false|
|movielens|     top_movies|      false|
+---------+---------------+-----------+



## 9. Verify Output

In [0]:
display(dbutils.fs.ls("/FileStore/movielens/processed/"))

path,name,size,modificationTime
dbfs:/FileStore/movielens/processed/cleaned/,cleaned/,0,1779546492000
dbfs:/FileStore/movielens/processed/top_per_genre/,top_per_genre/,0,1779546492000


## 10. Processing Summary

| Metric | Value |
|---|---|
| Input records | 25,062,518 |
| After cleaning + explode | 67,809,886 |
| Duplicate pairs removed | 0 |
| Movie-genre combinations | 64,473 |
| Distinct genres | 20 |
| Top-N per genre | 10 |
| Hive tables | `movielens.ratings_cleaned`, `movielens.top_movies` |


Proceed to Notebook 05 — Data Analysis and AI.